In [ ]:
!pip install bitsandbytes

In [ ]:
!pip install -q accelerate peft  transformers trl

In [ ]:
#clear working directory
import shutil
import os

# Path to the working directory (adjust if needed)
working_dir = '/kaggle/working/'

# Remove all contents of the working directory
for item in os.listdir(working_dir):
    item_path = os.path.join(working_dir, item)
    if os.path.isfile(item_path) or os.path.islink(item_path):
        os.unlink(item_path)
    elif os.path.isdir(item_path):
        shutil.rmtree(item_path)

print("Working directory cleared.")

In [ ]:
#load model and set lora configurations as well as hyper paramters
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer
model_name = "/kaggle/input/the_naswail_autopilot/maxtext/default/3/mergedfinelamaboi"

# The instruction dataset to use
dataset_name = "dataset.txt"

# Fine-tuned model name
new_model = "extra_fidfggg"

################################################################################
# QLoRA parameters
################################################################################

# LoRA attention dimension
lora_r = 64

# Alpha parameter for LoRA scaling
lora_alpha = 16

# Dropout probability for LoRA layers
lora_dropout = 0.1

################################################################################
# bitsandbytes parameters
################################################################################

# Activate 4-bit precision base model loading
use_4bit = True

# Compute dtype for 4-bit base models
bnb_4bit_compute_dtype = "float16"

# Quantization type (fp4 or nf4)
bnb_4bit_quant_type = "nf4"

# Activate nested quantization for 4-bit base models (double quantization)
use_nested_quant = True

################################################################################
# TrainingArguments parameters
################################################################################

# Output directory where the model predictions and checkpoints will be stored
output_dir = "./likesmen"

# Number of training epochs
num_train_epochs = 2

# Enable fp16/bf16 training (set bf16 to True with an A100)
fp16 = False
bf16 = False

# Batch size per GPU for training
per_device_train_batch_size = 1

# Batch size per GPU for evaluation
per_device_eval_batch_size = 1

# Number of update steps to accumulate the gradients for
gradient_accumulation_steps = 1

# Enable gradient checkpointing
gradient_checkpointing = True

# Maximum gradient normal (gradient clipping)
max_grad_norm = 0.3

# Initial learning rate (AdamW optimizer)
learning_rate = 2e-4

# Weight decay to apply to all layers except bias/LayerNorm weights
weight_decay = 0.001

# Optimizer to use
optim = "paged_adamw_32bit"

# Learning rate schedule
lr_scheduler_type = "cosine"

# Number of training steps (overrides num_train_epochs)
max_steps = -1

# Ratio of steps for a linear warmup (from 0 to learning rate)
warmup_ratio = 0.03

# Group sequences into batches with same length
# Saves memory and speeds up training considerably
group_by_length = True

# Save checkpoint every X updates steps
save_steps = 0

# Log every X updates steps
logging_steps = 25

################################################################################
# SFT parameters
################################################################################

# Maximum sequence length to use
max_seq_length = None

# Pack multiple short examples in the same input sequence to increase efficiency
packing = False

# Load the entire model on the GPU 0
device_map = "auto"  # Let Hugging Face automatically split layers across GPUs

In [ ]:
#train model(olderboi savees after each epoch)



# Load dataset (you can process it here)
# Load dataset
dataset = load_dataset('text', data_files=dataset_name, split="train")

# Split into train (80%) and validation (20%)
split_dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(f"Train size: {len(train_dataset)}, Eval size: {len(eval_dataset)}")


# Load tokenizer and model with QLoRA configuration
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

# Check GPU compatibility with bfloat16
if compute_dtype == torch.float16 and use_4bit:
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print("=" * 80)
        print("Your GPU supports bfloat16: accelerate training with bf16=True")
        print("=" * 80)

# Load base model
# Force model to use single GPU
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
      # Change this to
    device_map="auto"  # Force single GPU
)
model.config.use_cache = False
model.config.pretraining_tp = 1

# Load LLaMA tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Fix weird overflow issue with fp16 training

# Load LoRA configuration
peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM",
)

# Set training parameters
training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    evaluation_strategy = "steps",  # or "epoch" to evaluate at the end of each epoch
    eval_steps = 50,                 # evaluate every 50 steps
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    lr_scheduler_type=lr_scheduler_type,
    report_to="tensorboard"
                   # Only keep the best model
)

# Set supervised fine-tuning parameters
# Define a formatting function to extract the 'text' field from each example
def formatting_func(example):
    return example['text']

# Create a data collator with your tokenizer
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Causal language modeling (not masked LM)
)

# Update SFTTrainer initialization
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,      
    eval_dataset=eval_dataset,        
    peft_config=peft_config,
    formatting_func=formatting_func,
    args=training_arguments,
    data_collator=data_collator,
)

# Train model
trainer.train()
trainer.model.save_pretrained(new_model)

In [ ]:
trainer.model.save_pretrained(new_model)
#del new_model

In [ ]:
#merge model gpu

#save model
#trainer.model.save_pretrained(new_model)
# Reload model in FP16 and merge it with LoRA weights
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer
new_model="extra_fidfggg"
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)
model_name="/kaggle/input/the_naswail_autopilot/maxtext/default/3/mergedfinelamaboi"
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True,
    return_dict=True,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, new_model)
merged_model = model.merge_and_unload()

# Reload tokenizer to save it
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
# save merged model
save_dir = "/kaggle/working/mergedfinelamaboi"

# Save merged model
merged_model.save_pretrained(
    save_dir,
    safe_serialization=True,  # Use safetensors format (recommended)
    max_shard_size="2GB"  # Split into smaller files
)

# Save tokenizer
tokenizer.save_pretrained(save_dir)


In [ ]:
#half of testing (only run once)

import os
import torch
import time  # Import the time module
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

# Start measuring runtime


# Quantization configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)
#model_boi2="Orenguteng/Llama-3.1-8B-Lexi-Uncensored-V2"#(for testing the diffrence between the base model and the mergerd)
#model_boi = "/kaggle/working/mergedfinelamaboi"#actual merged model
model_boi="/kaggle/input/the_naswail_autopilot/maxtext/default/3/mergedfinelamaboi"#remove later
# Load quantized model
model = AutoModelForCausalLM.from_pretrained(
    model_boi,
    #quantization_config=quantization_config,#the merged model is already quantized so keep this commented however for testing base model uncomment it
    device_map="auto"
     
).eval()

tokenizer = AutoTokenizer.from_pretrained(model_boi)

In [ ]:
#actual testing
start_time = time.time()
# Ignore warnings
logging.set_verbosity(logging.CRITICAL)

sysprompt = """<|start_header_id|>system<|end_header_id|>  
You are a cybersecurity assistant that provides responses in JSON format.  
<|eot_id|> """
jason_prompt="Respond in JSON format only with two things function:name of the function(of which there is block_ip() and block_port() and terminate_process() and set_rate_limit only) and paramter:the paramter you were trained to produce which in the case of set_rate_limit it has paramter and parameter2, no additional text."
# Run text generation pipeline with our next model
prompt = """<|start_header_id|>user<|end_header_id|>  
Rate limiting serves as a gatekeeper at the network edge, ensuring that traffic from each source does not exceed a predefined rate threshold OWASP Cheat Sheet Series. Implement Firewalls:Firewalls can help prevent DoS attacks by blocking traffic from known malicious IP addresses or by limiting the amount of traffic allowed from a single sourc limiting the amount of traffic allowed from a single source. Firewalls and routers, when configured with robust ingress and egress filtering practices, can prevent devices from becoming unwitting soldiers in a botnet army and block traffic limiting the amount of traffic allowed from a single source. Firewalls and routers, when configured with robust ingress and egress filtering practices, can prevent devices from becoming unwitting soldiers in a botnet army and block traffic from known malicious sources. Rate-limiting, employed at both hardware and software levels, acts as a regulator for the volume of traffic or the number of requests, allowing for measures such as traffic shaping and deep packet inspection to maintain order within the network. Limit Bandwidth:Implementing bandwidth limitations on incoming traffic can help prevent a DoS attack from overwhelming the network or server. These systems can analyse traffic over time, establishing a baseline and identifying abnormal patterns that could suggest an ongoing attack. Network Segmentation:Segmenting the network can help prevent a DoS attack from spreading throughout the entire network. Patching these vulnerabilities can prevent a DoS attack from being successful. Following is the command for performing flooding of requests on an IP. For example, if a bank website can handle 10 people a second by clicking the Login button, an attacker only has to send 10 fake requests per second to make it so no legitimate users can log in. Traffic Anomalies and Performance Degradation
Traffic anomalies and performance degradation are the way. 172.16.254.1 34567  
.
"""

pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=500)
result = pipe(sysprompt +jason_prompt+ prompt)

print(result[0]['generated_text'])

# End measuring runtime
end_time = time.time()
total_time = end_time - start_time
print(f"Total runtime: {total_time:.2f} seconds")